In [1]:
%reset -f
%reload_ext autoreload
%autoreload 2

In [2]:
import os,sys
current_path = os.getcwd()
sys.path.append("/home/zhuchen/poc/scorecard") 

In [3]:
name = current_path.split('/')[-1].split('_')
data = name[0]
cust = name[1]

In [4]:
print(name)

['deltaV1', 'all']


In [5]:
FILE_PATH = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/file/'
DATA_PATH = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/data/'
TMP_PATH  = f'/home/zhuchen/poc/02-xyf/{data}_{cust}/tmp/'
DATA_OR_PATH = '/home/zhuchen/poc/02-xyf/data_or/'

In [6]:
import gc
import pandas as pd 
import numpy as np  
import math  
import matplotlib.pyplot as plt
import copy
import zipfile
import seaborn as sns
from pylab import mpl
from ScoreCard.creat_report import Report
mpl.rcParams['font.sans-serif'] = ['SimHei']
mpl.rcParams['axes.unicode_minus'] = False  

isExists=os.path.exists(FILE_PATH)
if not isExists:
    os.makedirs(FILE_PATH) 
    
isExists=os.path.exists(DATA_PATH)
if not isExists:
    os.makedirs(DATA_PATH)

isExists=os.path.exists(TMP_PATH)
if not isExists:
    os.makedirs(TMP_PATH)  

import warnings
warnings.filterwarnings("ignore")

In [7]:
if cust == 'all':
    tag = ''
else:
    tag = '_'+cust
or_file_name = f'{data}_data_all1{tag}.parquet'

In [ ]:
if data == 'deltaV1' and cust == 'all':
    deltaV1_total_iv = pd.read_csv(f'/home/zhuchen/poc/02-xyf/data_or/deltaV1_iv.csv')
    deltaV1_total_iv.sort_values(['total_iv'],ascending=False,inplace=True)
    iv_list = deltaV1_total_iv['var_name'].to_list()[0:2000]
    data_all1 = pd.read_parquet(DATA_OR_PATH + or_file_name,columns = ['mobile', 'backPointTime', 'qudao3_act', 'label'] +iv_list)
elif data == 'deltaV1' and cust == 'api':
    deltaV1_total_iv = pd.read_csv(f'/home/zhuchen/poc/02-xyf/data_or/deltaV1_api_iv.csv')
    deltaV1_total_iv.sort_values(['total_iv'],ascending=False,inplace=True)
    iv_list = deltaV1_total_iv['var_name'].to_list()[0:3500]
    data_all1 = pd.read_parquet(DATA_OR_PATH + or_file_name,columns = ['mobile', 'backPointTime', 'qudao3_act', 'label'] +iv_list)
else:
    data_all1 = pd.read_parquet(DATA_OR_PATH + or_file_name)

In [9]:
data_all1.head()

,mobile,backPointTime,qudao3_act,label,TZ_1381_m12,TZ_1381_m24,TZ_1381_m18,TZ_0264_m15,TZ_0264_m12,TZ_0264_m18,...,TZ_0244_m4,TZ_0075_w4,TZ_0418_m8,TZ_0033_m3,TZ_0418_m9,TZ_0443_m8,TZ_0455_m3,TZ_0418_m11,TZ_0269_w4,TZ_0049_m12
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API,0.0,-1.0,10.0,-1.0,1.0,1.0,1.0,...,0.0,0.000000,-1.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API,0.0,20.0,20.0,20.0,35.0,35.0,35.0,...,8.0,0.916667,1.0,1.0,1.0,-1.0,3.0,1.0,0.0,0.0
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API,0.0,18.0,20.0,20.0,20.0,18.0,20.0,...,-1.0,0.333333,1.0,2.0,1.0,-1.0,-1.0,1.0,2.0,0.0


In [10]:
data_all1.shape

(1277742, 2004)

### 1.1 去重 & 去空

In [11]:
data_all1.shape

(1277742, 2004)

In [12]:
data_all1['label'].isnull().sum()

0

In [13]:
data_all1 = data_all1[~data_all1['label'].isnull()].reset_index(drop=True)

In [14]:
data_all1

,mobile,backPointTime,qudao3_act,label,TZ_1381_m12,TZ_1381_m24,TZ_1381_m18,TZ_0264_m15,TZ_0264_m12,TZ_0264_m18,...,TZ_0244_m4,TZ_0075_w4,TZ_0418_m8,TZ_0033_m3,TZ_0418_m9,TZ_0443_m8,TZ_0455_m3,TZ_0418_m11,TZ_0269_w4,TZ_0049_m12
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API,0.0,-1.0,10.0,-1.0,1.0,1.0,1.0,...,0.0,0.000000,-1.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API,0.0,20.0,20.0,20.0,35.0,35.0,35.0,...,8.0,0.916667,1.0,1.0,1.0,-1.0,3.0,1.0,0.0,0.0
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API,0.0,18.0,20.0,20.0,20.0,18.0,20.0,...,-1.0,0.333333,1.0,2.0,1.0,-1.0,-1.0,1.0,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1277737,a18e8ccd88bb61bc7f9dd825d25b8850,2024-11-13,APP自然流量,0.0,24.0,31.0,30.0,27.0,25.0,31.0,...,-1.0,0.666667,1.0,0.0,1.0,0.0,-1.0,1.0,2.0,0.0
1277738,e0c1b14d5f5953670ed63c102f3ead9f,2024-11-15,APP自然流量,0.0,-1.0,-1.0,-1.0,20.0,20.0,20.0,...,-1.0,0.000000,1.0,2.0,1.0,1.0,-1.0,1.0,-1.0,1.0
1277739,82bc32734d6acd2f8993add147648f26,2024-11-12,APP自然流量,0.0,25.0,38.0,25.0,26.0,26.0,26.0,...,0.0,0.250000,1.0,1.0,1.0,1.0,0.0,1.0,-1.0,1.0
1277740,45ed4aab69e0a3039def1ed484e79716,2024-11-23,APP自然流量,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,20.0,0.000000,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0


In [15]:
data_all1[['mobile','backPointTime','qudao3_act']].value_counts()

mobile                            backPointTime  qudao3_act
aa2cd2b58fbd554c6bd30187ff5a2e77  2024-11-25     信息流           2
b11611521b5059ffabe19b71132a4c45  2024-11-04     信息流           2
1f96bb61f51b898a87bb547c0637920d  2025-05-19     信息流           2
c4851c5a6baec545cf67a86cd7c471df  2024-08-09     信息流           2
d7cfe754b27299180947d90b10215db4  2025-04-07     信息流           2
                                                              ..
54eb9b9b8d8a2c6d6488f00eec284d21  2025-06-19     信息流           1
54eb9a32b5c6224ca3e2e7adb38597d9  2025-01-26     信息流           1
54eb939869e2a2e00d3ac252250b616f  2025-01-13     API           1
54eb8f55a4dab82bbb2e856c83f54b7e  2024-10-02     API           1
fffffc381ac87676d9c4cad3dfc9972d  2024-09-17     APP自然流量       1
Length: 1277540, dtype: int64

In [16]:
(data_all1[['mobile','backPointTime']].value_counts() > 1).sum()

202

In [17]:
data_all1.shape

(1277742, 2004)

In [18]:
# data_all1[(data_all1['mobile_sha256'] == '8981c912918c45aa4693b8e042bf42bc6c2c8c0938931696efe1e5c99df53bc8') & (data_all1['hs_date'] == '2024-10-26')].to_csv('data_all1_2024-10-26.csv', index=False)

In [19]:
# 这一步老大的意思是可以不去重
# data_all1.drop_duplicates(['mobile_sha256','hs_date','sample_type'],inplace=True)
# data_all1.shape

In [20]:
# data_all1[['mobile_sha256','hs_date','sample_type']].value_counts()

In [21]:
# data_all1[['mobile_sha256','hs_date','sample_type']].value_counts().iloc[0]

### 1.2 数据类型转换

In [22]:
data_all1.select_dtypes(include='object')

,mobile,backPointTime,qudao3_act
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API
...,...,...,...
1277737,a18e8ccd88bb61bc7f9dd825d25b8850,2024-11-13,APP自然流量
1277738,e0c1b14d5f5953670ed63c102f3ead9f,2024-11-15,APP自然流量
1277739,82bc32734d6acd2f8993add147648f26,2024-11-12,APP自然流量
1277740,45ed4aab69e0a3039def1ed484e79716,2024-11-23,APP自然流量


### 1.3 观察数据时间分布

In [23]:
gc.collect()

0

In [24]:
from DataPreprocessing.Datasets.Describe import badrate_by_month
# data_all1.reset_index(drop=True, inplace=True)
data_all1['target'] = 'train'

In [25]:
data_all1.head()

,mobile,backPointTime,qudao3_act,label,TZ_1381_m12,TZ_1381_m24,TZ_1381_m18,TZ_0264_m15,TZ_0264_m12,TZ_0264_m18,...,TZ_0075_w4,TZ_0418_m8,TZ_0033_m3,TZ_0418_m9,TZ_0443_m8,TZ_0455_m3,TZ_0418_m11,TZ_0269_w4,TZ_0049_m12,target
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0,train
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.250000,-1.0,0.0,-1.0,0.5,8.0,-1.0,0.0,0.0,train
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API,0.0,-1.0,10.0,-1.0,1.0,1.0,1.0,...,0.000000,-1.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0,train
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API,0.0,20.0,20.0,20.0,35.0,35.0,35.0,...,0.916667,1.0,1.0,1.0,-1.0,3.0,1.0,0.0,0.0,train
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API,0.0,18.0,20.0,20.0,20.0,18.0,20.0,...,0.333333,1.0,2.0,1.0,-1.0,-1.0,1.0,2.0,0.0,train


In [26]:
month_time = badrate_by_month(data_all1, 'backPointTime', ['train'], 'label')
month_time

,count,bad,good,bad_rate
2024-07,101890,3192.0,98698.0,0.031328
2024-08,111257,4113.0,107144.0,0.036968
2024-09,116716,4736.0,111980.0,0.040577
2024-10,113623,5003.0,108620.0,0.044032
2024-11,105345,4880.0,100465.0,0.046324
2024-12,107385,4970.0,102415.0,0.046282
2025-01,120569,5661.0,114908.0,0.046952
2025-02,103627,5473.0,98154.0,0.052814
2025-03,114568,6419.0,108149.0,0.056028
2025-04,109790,6019.0,103771.0,0.054823


In [27]:
# plt.figure(figsize=(10, 5))
# plt.plot(month_time.index, month_time['bad_rate'], linewidth=2, label='bad_rate')
# plt.title('bad_rate by month', fontsize=15)
# plt.xlabel("time", fontsize=12)
# plt.ylabel("bad_rate", fontsize=12)
# plt.tick_params(axis='both', labelsize=10)
# plt.ylim(min(month_time['bad_rate'])*0.9,max(month_time['bad_rate'])*1.1)
# plt.show()

In [28]:
from DataPreprocessing.Datasets.Describe import badrate_by_day
day_time = badrate_by_day(data_all1, 'backPointTime', ['train'], 'label')
day_time

,count,bad,good,bad_rate
2024-07-01,3397,104.0,3293.0,0.030615
2024-07-02,3351,112.0,3239.0,0.033423
2024-07-03,3514,106.0,3408.0,0.030165
2024-07-04,3504,109.0,3395.0,0.031107
2024-07-05,3739,97.0,3642.0,0.025943
...,...,...,...,...
2025-06-18,2783,161.0,2622.0,0.057851
2025-06-19,2946,167.0,2779.0,0.056687
2025-06-20,2976,161.0,2815.0,0.054099
2025-06-21,2235,108.0,2127.0,0.048322


In [29]:
# plt.figure(figsize=(15, 5))
# plt.plot(day_time.index, day_time['bad_rate'], linewidth=2, label='bad_rate')
# plt.title('bad_rate by day', fontsize=15)
# plt.xlabel("time", fontsize=12)
# plt.ylabel("bad_rate", fontsize=12)
# plt.tick_params(axis='both', labelsize=10)
# plt.ylim(min(day_time['bad_rate'])*0.8,max(day_time['bad_rate'])*1.1)
# plt.show()

In [30]:
from DataPreprocessing.Datasets.Describe import badrate_by_week
week_time = badrate_by_week(data_all1, 'backPointTime', ['train'], 'label')
week_time

,count,bad,good,bad_rate,date_max,date_min
1,26765,1217.0,25548.0,0.045470,2025-01-05,2024-12-30
2,26004,1223.0,24781.0,0.047031,2025-01-12,2025-01-06
3,26256,1246.0,25010.0,0.047456,2025-01-19,2025-01-13
4,33055,1547.0,31508.0,0.046801,2025-01-26,2025-01-20
5,22398,1104.0,21294.0,0.049290,2025-02-02,2025-01-27
6,27266,1443.0,25823.0,0.052923,2025-02-09,2025-02-03
7,25908,1372.0,24536.0,0.052957,2025-02-16,2025-02-10
8,25281,1341.0,23940.0,0.053044,2025-02-23,2025-02-17
9,26265,1363.0,24902.0,0.051894,2025-03-02,2025-02-24
10,27990,1602.0,26388.0,0.057235,2025-03-09,2025-03-03


In [31]:
# plt.figure(figsize=(15, 5))
# plt.plot(week_time.index, week_time['bad_rate'], linewidth=2, label='bad_rate')
# plt.title('bad_rate by week', fontsize=15)
# plt.xlabel("time", fontsize=12)
# plt.ylabel("bad_rate", fontsize=12)
# plt.tick_params(axis='both', labelsize=10)
# plt.ylim(0,max(week_time['bad_rate'])*1.1)
# plt.show()

### 1.4 分割数据集

In [32]:
data_all1.head()

,mobile,backPointTime,qudao3_act,label,TZ_1381_m12,TZ_1381_m24,TZ_1381_m18,TZ_0264_m15,TZ_0264_m12,TZ_0264_m18,...,TZ_0418_m9,TZ_0443_m8,TZ_0455_m3,TZ_0418_m11,TZ_0269_w4,TZ_0049_m12,target,month_time,day_time,week_time
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.5,8.0,-1.0,0.0,0.0,train,2024-08,2024-08-11,32
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.5,8.0,-1.0,0.0,0.0,train,2024-08,2024-08-11,32
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API,0.0,-1.0,10.0,-1.0,1.0,1.0,1.0,...,-1.0,-1.0,0.0,-1.0,-1.0,0.0,train,2025-02,2025-02-21,8
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API,0.0,20.0,20.0,20.0,35.0,35.0,35.0,...,1.0,-1.0,3.0,1.0,0.0,0.0,train,2025-06,2025-06-09,24
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API,0.0,18.0,20.0,20.0,20.0,18.0,20.0,...,1.0,-1.0,-1.0,1.0,2.0,0.0,train,2025-04,2025-04-28,18


In [33]:
data_all1['weight'] = 1
data_all1['target'] = 'train'

In [ ]:
# 最后10%为oot
data_all1.loc[(pd.to_datetime(data_all1['backPointTime']) >= pd.to_datetime(data_all1['backPointTime']).quantile(0.9)) & (data_all1['target'] =='train'),'target'] = 'oot'

In [35]:
# data_all1.loc[((data_all1['stage']) == 'test') & (data_all1['target'] =='train'),'target'] = 'oot'

In [36]:
data_all1['target'].value_counts()

train    1148251
oot       129491
Name: target, dtype: int64

In [37]:
data_all1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1277742 entries, 0 to 1277741
Columns: 2009 entries, mobile to weight
dtypes: float32(2000), float64(1), int64(2), object(6)
memory usage: 9.6+ GB


In [38]:
gc.collect()

0

In [39]:
from DataPreprocessing.Datasets.Split import by_stratify
data_all1, res = by_stratify(data_all1, 0.3, 2025, date='backPointTime', dep='label')

         count  bad_rate      bad    time_min    time_max
all    1277742  0.046695  59664.0  2024-07-01  2025-06-22
train   803775  0.046010  36982.0  2024-07-01  2025-05-10
valid   344476  0.046009  15849.0  2024-07-01  2025-05-10
oot     129491  0.052768   6833.0  2025-05-11  2025-06-22
ratio_bad_good:  0.04822944393076097
train    803775
valid    344476
oot      129491
Name: target, dtype: int64


In [40]:
data_all1.head()

,mobile,backPointTime,qudao3_act,label,TZ_1381_m12,TZ_1381_m24,TZ_1381_m18,TZ_0264_m15,TZ_0264_m12,TZ_0264_m18,...,TZ_0443_m8,TZ_0455_m3,TZ_0418_m11,TZ_0269_w4,TZ_0049_m12,target,month_time,day_time,week_time,weight
0,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,8.0,-1.0,0.0,0.0,valid,2024-08,2024-08-11,32,1
1,663c8e4d4a0d11fa9767b332f30c1921,2024-08-11,信息流,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,8.0,-1.0,0.0,0.0,train,2024-08,2024-08-11,32,1
2,f3f4e2f3c61d8f1a035a0c32987714a5,2025-02-21,API,0.0,-1.0,10.0,-1.0,1.0,1.0,1.0,...,-1.0,0.0,-1.0,-1.0,0.0,valid,2025-02,2025-02-21,8,1
3,6d09e74a845b0c12e86be848d40f4fd3,2025-06-09,API,0.0,20.0,20.0,20.0,35.0,35.0,35.0,...,-1.0,3.0,1.0,0.0,0.0,oot,2025-06,2025-06-09,24,1
4,70a24bd8e0cfa4efb16a5a26b3054bc5,2025-04-28,API,0.0,18.0,20.0,20.0,20.0,18.0,20.0,...,-1.0,-1.0,1.0,2.0,0.0,train,2025-04,2025-04-28,18,1


In [41]:
res

,count,bad_rate,bad,time_min,time_max
all,1277742,0.046695,59664.0,2024-07-01,2025-06-22
train,803775,0.046010,36982.0,2024-07-01,2025-05-10
valid,344476,0.046009,15849.0,2024-07-01,2025-05-10
oot,129491,0.052768,6833.0,2025-05-11,2025-06-22


In [127]:
# data_all1.to_pickle(DATA_PATH + 'data_model.pkl')
# data_all1.to_pickle(DATA_PATH + 'data_model.zip', compression='zip')

In [42]:
data_all1.to_parquet(DATA_PATH + 'data_model.parquet')

### 1.5 数据分析

In [77]:
# data_all1 = pd.read_parquet(DATA_PATH + 'data_model.parquet')

In [43]:
ex_lst = ['mobile','backPointTime','qudao3_act','label',
 'weight',
 'target',
 'month_time',
 'day_time',
 'week_time',]
ft_lst = [i for i in data_all1.columns if i not in ex_lst]
len(ft_lst)

2000

In [44]:
print(ft_lst)

['TZ_1381_m12', 'TZ_1381_m24', 'TZ_1381_m18', 'TZ_0264_m15', 'TZ_0264_m12', 'TZ_0264_m18', 'TZ_0264_m24', 'TZ_0266_m24', 'TZ_0266_m15', 'TZ_0266_m18', 'TZ_0266_m12', 'TZ_0266_m9', 'TZ_0264_m9', 'TZ_0272_m24', 'TZ_0326_m24', 'TZ_0080_m24', 'TZ_0454_m24', 'TZ_0083_m24', 'TZ_1381_m6', 'TZ_0265_m15', 'TZ_0267_m24', 'TZ_0324_m24', 'TZ_0265_m18', 'TZ_0265_m24', 'TZ_0265_m12', 'TZ_0266_m6', 'TZ_0267_m18', 'TZ_0081_m24', 'TZ_1317_m24', 'TZ_0272_m18', 'TZ_0274_m24', 'TZ_0266_m5', 'TZ_0272_m15', 'TZ_0265_m9', 'TZ_0080_m18', 'TZ_0326_m18', 'TZ_0083_m18', 'TZ_0264_m6', 'TZ_0080_m15', 'TZ_0325_m24', 'TZ_0267_m15', 'TZ_1317_m18', 'TZ_0083_m15', 'TZ_0324_m18', 'TZ_0326_m15', 'TZ_1317_m12', 'TZ_0267_m12', 'TZ_0273_m24', 'TZ_0081_m18', 'TZ_0267_m9', 'TZ_0080_m12', 'TZ_0324_m15', 'TZ_0083_m12', 'TZ_0272_m12', 'TZ_0081_m15', 'TZ_0264_m5', 'TZ_0274_m18', 'TZ_0266_m4', 'TZ_0265_m6', 'TZ_0454_m12', 'TZ_0274_m15', 'TZ_0081_m12', 'TZ_0326_m12', 'TZ_0327_m24', 'TZ_0325_m18', 'TZ_0273_m18', 'TZ_0324_m12', 'TZ_0

In [45]:
gc.collect()

0

In [46]:
from ScoreCard.creat_report import Report
length = 300
for i in range(0,(len(ft_lst) // length) + 1):
    start, end = i * length, min((i + 1) * length, len(ft_lst))
    print(start, ' - ', end)
    report = Report(data_all1[ft_lst[start:end] + ex_lst], ex_lst, 'label', FILE_PATH + str(i) + '_')
#     key_value, bins_detail = report.iv_ks_psi()
#     key_value, bins_detail = report.iv_ks_psi_chi2()
    key_value, bins_detail = report.iv_ks_psi_chi2_missing_quantile()
    del report,key_value,bins_detail 

0  -  300
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
300  -  600
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
600  -  900
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
900  -  1200
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
1200  -  1500
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
1500  -  1800
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存
1800  -  2000
正在计算IV......
正在计算CHI2......
正在计算COVER RATE......
正在计算QUANTILE......
正在保存结果......
已保存


In [47]:
import os
files_key_value, files_bins_detail = [], []
for root, dirs, files in os.walk(FILE_PATH[:-1]): 
    for file in files:
        if '_bins_detail' in file:
            files_bins_detail.append(pd.read_excel(root +'/'+ file))
        elif 'iv_ipt_psi_chi2_missing' in file:
            files_key_value.append(pd.read_excel(root +'/'+ file))
            
key_value = pd.concat(files_key_value)
bins_detail = pd.concat(files_bins_detail)

bins_detail.to_excel(FILE_PATH + 'bins_detail.xlsx')
key_value.to_excel(FILE_PATH + 'key_value.xlsx')

In [48]:
key_value.sort_values(by='iv_bin_train', ascending=False)

,Unnamed: 0,var_names,ks,iv_bin_train,iv_bin_valid,iv_bin_oot,iv_bin_total,psi_tv_bin,psi_to_bin,psi_tvo_bin,...,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99,max
24,24,TZ_1381_m12,0.0,0.038564,0.036497,0.008511,0.032865,1.108571e-06,0.015145,0.015099,...,-1.0,-1.0,-1.0,-1.0,2.0,8.0,20.0,25.000000,38.000000,56.000000
227,227,TZ_1381_m24,0.0,0.037503,0.038238,0.011769,0.032295,1.339798e-05,0.009230,0.009091,...,-1.0,-1.0,-1.0,0.0,4.0,15.0,23.0,32.000000,39.000000,56.000000
206,206,TZ_1381_m18,0.0,0.037476,0.036241,0.012913,0.032099,1.957952e-05,0.015288,0.015252,...,-1.0,-1.0,-1.0,0.0,3.0,12.0,22.0,29.000000,38.000000,56.000000
202,202,TZ_0264_m24,0.0,0.036290,0.035740,0.008300,0.029613,3.680588e-06,0.032093,0.032057,...,-1.0,1.0,4.0,10.0,20.0,23.0,33.0,38.000000,44.000000,56.000000
204,204,TZ_0264_m12,0.0,0.035987,0.036751,0.011556,0.029685,1.010345e-05,0.042098,0.042134,...,-1.0,0.0,2.0,5.0,11.0,20.0,24.0,31.000000,39.000000,56.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,84,TZ_0455_m3,0.0,0.003567,0.004010,0.003825,0.003395,1.111656e-07,0.008236,0.008251,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,3.000000,17.000000,56.000000
41,41,TZ_0288_m4,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.000000,2.000000,18.000000
40,40,TZ_0290_m4,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.000000,2.000000,4.000000
174,174,TZ_0143_m3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.333333,110.333336


In [49]:
bins_detail.sort_values(by='total_iv', ascending=False)

,Unnamed: 0,Unnamed: 1,variable,value,bin,count,count_distr,good,goodprob,bad,badprob,iv,total_iv,ks,total_ks,woe
1,NaN,1,TZ_1381_m12,1,"[0.5, 4.5)",80579,0.100251,76879,0.100260,3700,0.100049,4.477439e-07,0.038564,0.073029,0.073241,-0.002114
0,TZ_1381_m12,0,TZ_1381_m12,0,"[-inf, 0.5)",534706,0.665243,507520,0.661874,27186,0.735114,7.686742e-03,0.038564,0.073241,0.073241,0.104952
2,NaN,2,TZ_1381_m12,2,"[4.5, 17.5)",80474,0.100120,77495,0.101064,2979,0.080553,4.652737e-03,0.038564,0.052518,0.073241,-0.226840
3,NaN,3,TZ_1381_m12,3,"[17.5, 21.5)",42969,0.053459,41603,0.054256,1366,0.036937,6.659137e-03,0.038564,0.035199,0.073241,-0.384500
4,NaN,4,TZ_1381_m12,4,"[21.5, inf)",65047,0.080927,63296,0.082546,1751,0.047347,1.956537e-02,0.038564,0.000000,0.073241,-0.555850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,TZ_0455_m3,0,TZ_0455_m3,0,"[-inf, -0.5)",666622,0.829364,635709,0.829049,30913,0.835893,5.626855e-05,0.003567,0.006844,0.012769,0.008221
581,TZ_0289_m4,0,TZ_0289_m4,0,"[-inf, inf)",803775,1.000000,766793,1.000000,36982,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
580,TZ_0288_m4,0,TZ_0288_m4,0,"[-inf, inf)",803775,1.000000,766793,1.000000,36982,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
579,TZ_0290_m4,0,TZ_0290_m4,0,"[-inf, inf)",803775,1.000000,766793,1.000000,36982,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000


In [50]:
%reset -f